# Türkçe Akademik NLP Studio (Python / Jupyter)
HTML aracının Python karşılığı: **metin sütunu seçimi** • LDA • Transformer embedding • HDBSCAN • BERTopic zinciri (embedding → HDBSCAN → c-TF-IDF) • NER • duygu analizi • JSON/CSV çıktı.

Notlar:
- LDA, HTML'deki Gibbs örneklemesi yerine `scikit-learn` varyasyonel LDA ile çalışır (büyük korpusta çok daha hızlıdır).
- HDBSCAN, `scikit-learn`'ün resmi `HDBSCAN` sınıfıyla çalışır (HTML'deki yaklaşık bileşen tabanlı sürümden daha doğrudur).
- Transformer modelleri ilk çalıştırmada Hugging Face'ten indirilir (GPU varsa otomatik kullanılır).

In [ ]:
# Gerekli paketler (Colab / yeni ortam için bir kez çalıştırın)
%pip install -q pandas numpy scikit-learn>=1.3 sentence-transformers transformers torch matplotlib

In [ ]:
import re, json, math, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---------------- AYARLAR ----------------
CSV_PATH      = "hr_dergi_3649_guncel.csv"   # .csv / .tsv / .zip (içinde csv) olabilir
TEXT_COLUMN   = None      # None: otomatik seç (Metin > Dosyaİçeriği > _orig_text > Özet > en uzun metin sütunu)
ROW_MODE      = "rows"    # "rows": her satır ayrı belge | "all": sütunun tamamı tek korpus
MAX_DOCS      = 300       # None: tüm satırlar (LDA/embedding çok yavaşlayabilir)
RANDOM_SAMPLE = False     # True: rastgele örnekle, False: ilk N satır
SEED          = 42

# Ön işleme
MIN_LEN       = 2
USE_STOPWORDS = True
EXTRA_STOP    = []        # ör. ["kelime", "başka"]

# LDA
LDA_K, LDA_ITER, LDA_ALPHA, LDA_BETA = 5, 30, 0.10, 0.01

# HDBSCAN
MIN_CLUSTER, MIN_SAMPLES = 3, 2

# Modeller
EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
NER_MODEL   = "Davlan/bert-base-multilingual-cased-ner-hrl"
SENT_MODEL  = "nlptown/bert-base-multilingual-uncased-sentiment"
MAX_SENT, MAX_NER_SENT = 200, 100

## 1. Veriyi yükle ve metin sütununu seç

In [ ]:
def read_table(path):
    import io, zipfile
    if path.lower().endswith(".zip"):
        z = zipfile.ZipFile(path)
        name = next(n for n in z.namelist() if re.search(r"\.(csv|tsv|txt)$", n, re.I))
        print("ZIP içinden okunuyor:", name)
        raw = z.read(name)
    else:
        raw = open(path, "rb").read()
    first = raw[:20000].decode("utf-8-sig", "ignore").splitlines()[0]
    sep = max([",", ";", "\t"], key=first.count)          # ayraç tahmini (ilk satır)
    return pd.read_csv(io.BytesIO(raw), sep=sep, encoding="utf-8-sig")

df = read_table(CSV_PATH)
df.columns = [str(c).strip() for c in df.columns]
print(f"{len(df):,} satır • {df.shape[1]} sütun")

def col_stats(df, n=300):
    rows = []
    for c in df.columns:
        v = df[c].dropna().astype(str).head(n)
        avg = v.str.len().mean() if len(v) else 0
        num = v.str.fullmatch(r"-?[\d.,\s]+").mean() if len(v) else 1
        rows.append((c, round(avg), round(num, 2), (v.iloc[0][:70] if len(v) else "")))
    return pd.DataFrame(rows, columns=["sütun", "ort_uzunluk", "sayısal_oran", "örnek"])

stats = col_stats(df)
display(stats)

In [ ]:
def norm(s): return s.strip().lower().replace("İ","i").replace("I","ı")

def pick_text_column(stats):
    is_text = lambda r: r.ort_uzunluk >= 100 and r.sayısal_oran < .5
    cols = list(stats["sütun"])
    for pat in [r"^metin$", r"^(text|içerik|icerik)$", r"^dosyaiçeriği$", r"dosya\s*içeriği|metin|_orig_text"]:
        for c in cols:
            r = stats[stats["sütun"] == c].iloc[0]
            if re.search(pat, norm(c)) and is_text(r): return c
    for c in cols:
        r = stats[stats["sütun"] == c].iloc[0]
        if "özet" in norm(c) and is_text(r): return c
    t = stats[(stats.ort_uzunluk >= 100) & (stats.sayısal_oran < .5)].sort_values("ort_uzunluk", ascending=False)
    return t["sütun"].iloc[0] if len(t) else cols[0]

col = TEXT_COLUMN or pick_text_column(stats)
assert col in df.columns, f"Sütun bulunamadı: {col}"
print("Seçilen metin sütunu:", col)

vals = [x.strip() for x in df[col].dropna().astype(str) if x.strip()]
total = len(vals)
if ROW_MODE == "rows":
    if MAX_DOCS and len(vals) > MAX_DOCS:
        if RANDOM_SAMPLE:
            vals = list(np.random.default_rng(SEED).choice(vals, MAX_DOCS, replace=False))
        else:
            vals = vals[:MAX_DOCS]
    docs = vals
else:
    docs = ["\n".join(vals)]
print(f"{len(docs):,} belge analize alındı (kaynak: {total:,} dolu satır)")
print("Örnek:", docs[0][:300].replace("\n", " "), "…")

## 2. Ön işleme ve frekans

In [ ]:
STOP = set("""acaba ama ancak artık asla aslında az bazı bazıları bazen ben benim beri bile bir birçok biz bizim bu böyle bu yüzden da daha de değil diğer diye dolayısıyla en fakat gibi göre hangi hem henüz her hiç için ile ise işte kadar kendi kez ki kim mi mı mu mü nasıl ne neden nerede niçin o olan olarak ondan sonra onun veya ve ya ya da yani yine yok çünkü çok""".split())
EX = {norm(x) for x in EXTRA_STOP}

def tr_lower(s): return s.replace("İ","i").replace("I","ı").lower()

def tokenize(text):
    x = tr_lower(re.sub(r"\s+", " ", text))
    x = re.sub(r"\d+", " ", x)
    toks = re.findall(r"[^\W\d_]+", x, flags=re.UNICODE)
    if USE_STOPWORDS:
        toks = [w for w in toks if w not in STOP and w not in EX]
    return [w for w in toks if len(w) >= MIN_LEN]

doc_tokens = [tokenize(d) for d in docs]
all_tokens = [w for t in doc_tokens for w in t]
freq = Counter(all_tokens).most_common()
def split_sentences(t): return [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.strip()) > 15]
sentences = [s for d in docs for s in split_sentences(d)]

print(f"Belge: {len(docs):,} • Kelime: {len(all_tokens):,} • Benzersiz: {len(freq):,} • Cümle: {len(sentences):,}")
print(f"Ortalama belge uzunluğu: {len(all_tokens)/max(1,len(docs)):.1f} kelime")
top = freq[:20]
plt.figure(figsize=(10,4)); plt.bar([w for w,_ in top],[n for _,n in top],color="#315f8c")
plt.xticks(rotation=55, ha="right"); plt.title("En sık 20 kelime"); plt.tight_layout(); plt.show()
pd.DataFrame(freq[:100], columns=["Kelime","Frekans"]).head(30)

## 3. LDA konu modellemesi

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

cv = CountVectorizer(analyzer=lambda d: d, min_df=1)   # önceden tokenlanmış belgeler
X = cv.fit_transform(doc_tokens)
vocab = np.array(cv.get_feature_names_out())

lda = LatentDirichletAllocation(n_components=LDA_K, max_iter=LDA_ITER, doc_topic_prior=LDA_ALPHA,
                                topic_word_prior=LDA_BETA, learning_method="batch", random_state=SEED)
doc_topic = lda.fit_transform(X)
phi = lda.components_ / lda.components_.sum(1, keepdims=True)

lda_topics = []
for k in range(LDA_K):
    idx = phi[k].argsort()[::-1][:12]
    lda_topics.append([(vocab[i], float(phi[k][i])) for i in idx])
    print(f"Konu {k+1}: " + ", ".join(f"{vocab[i]} ({phi[k][i]:.3f})" for i in idx))
print("\nBelge başına baskın konu dağılımı:", dict(Counter(doc_topic.argmax(1)+1)))
print("Perplexity:", round(lda.perplexity(X), 1))

## 4. Transformer embedding

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
st_model = SentenceTransformer(EMBED_MODEL, device=device)
texts = [re.sub(r"\s+", " ", d)[:5000] for d in docs]
embeddings = st_model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
print("Embedding:", embeddings.shape, "• cihaz:", device)

## 5. HDBSCAN kümeleme

In [ ]:
from sklearn.cluster import HDBSCAN

hdb = HDBSCAN(min_cluster_size=max(2, MIN_CLUSTER), min_samples=max(1, MIN_SAMPLES), metric="euclidean")
labels = hdb.fit_predict(embeddings)    # embedding'ler L2-normalize olduğu için euclidean ≈ kosinüs
n_clusters = labels.max() + 1
print(f"Küme: {n_clusters} • Gürültü: {(labels < 0).sum()} • Belge: {len(docs)}")
for c in range(n_clusters):
    print(f"\n— Küme {c} ({(labels==c).sum()} belge)")
    for i in np.where(labels == c)[0][:5]:
        print("  •", i+1, docs[i][:150].replace("\n"," "))

## 6. BERTopic zinciri (embedding → HDBSCAN → c-TF-IDF)

In [ ]:
def ctfidf_for_clusters(labels, doc_tokens, top_n=10):
    n_docs = len(doc_tokens)
    df_count = Counter(w for t in doc_tokens for w in set(t))
    rows = []
    for c in range(labels.max() + 1):
        toks = [w for i in np.where(labels == c)[0] for w in doc_tokens[i]]
        N = len(toks) or 1
        sc = [(w, n / N * math.log((n_docs + 1) / (df_count.get(w, 1)))) for w, n in Counter(toks).items()]
        rows.append((c, sorted(sc, key=lambda x: -x[1])[:top_n]))
    return rows

bt = ctfidf_for_clusters(labels, doc_tokens)
for c, terms in bt:
    print(f"Topic {c}: " + ", ".join(f"{w} ({s:.4f})" for w, s in terms))

### (İsteğe bağlı) Resmi BERTopic paketi
Kurulum: `%pip install bertopic`. Aşağıdaki hücre yorum satırıdır; açarak UMAP + HDBSCAN + c-TF-IDF'li tam BERTopic'i çalıştırabilirsiniz.

In [ ]:
# from bertopic import BERTopic
# topic_model = BERTopic(embedding_model=st_model, min_topic_size=MIN_CLUSTER, language="multilingual")
# topics, probs = topic_model.fit_transform(texts, embeddings)
# topic_model.get_topic_info().head(15)

## 7. Duygu analizi

In [ ]:
from transformers import pipeline
dev = 0 if torch.cuda.is_available() else -1
sent_pipe = pipeline("sentiment-analysis", model=SENT_MODEL, device=dev)
sent_sentences = sentences[:MAX_SENT]
res = sent_pipe([s[:800] for s in sent_sentences], truncation=True, batch_size=16)
sent_df = pd.DataFrame({"Etiket":[r["label"] for r in res], "Güven":[round(r["score"],4) for r in res], "Cümle":[s[:180] for s in sent_sentences]})
print(sent_df["Etiket"].value_counts().to_string(), "\nOrtalama güven:", round(sent_df["Güven"].mean(),3))
sent_df.head(20)

## 8. Varlık adı tanıma (NER)

In [ ]:
ner_pipe = pipeline("token-classification", model=NER_MODEL, aggregation_strategy="simple", device=dev)
rows = []
for s in sentences[:MAX_NER_SENT]:
    for e in ner_pipe(s[:1200]):
        rows.append((e["entity_group"], e["word"], round(float(e["score"]),4), s[:160]))
ner_df = pd.DataFrame(rows, columns=["Varlık tipi","Varlık","Güven","Cümle"])
display(ner_df.groupby("Varlık tipi").size().rename("adet"))
ner_df.head(30)

## 9. Çıktıları kaydet

In [ ]:
report = {
    "source": {"column": col, "csv_rows": int(len(df)), "documents": len(docs)},
    "stats": {"documents": len(docs), "tokens": len(all_tokens), "unique": len(freq)},
    "frequency": freq[:500],
    "lda": {"K": LDA_K, "topics": lda_topics, "doc_topic": doc_topic.round(4).tolist()},
    "hdbscan": {"labels": labels.tolist(), "clusters": int(n_clusters)},
    "bertopic": [[c, t] for c, t in bt],
    "ner": ner_df.values.tolist(),
    "sentiment": sent_df.values.tolist(),
}
with open("turkce_nlp_studio_raporu.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o,"item") else str(o))
pd.DataFrame(freq, columns=["Kelime","Frekans"]).to_csv("turkce_kelime_frekans.csv", sep=";", index=False, encoding="utf-8-sig")
print("Kaydedildi: turkce_nlp_studio_raporu.json, turkce_kelime_frekans.csv")